In [1]:
#==============================================================================
# Loading the data
#==============================================================================

import pandas as pd

url = "https://raw.githubusercontent.com/Amirhossein-momeni/AmirGeek_ML_Datasets/refs/heads/main/DataSet/01-Ad.csv"
df = pd.read_csv(url)

In [22]:
#==============================================================================
# Exploring the data (EDA)
#==============================================================================

print(70* "-")  
print(df.shape) 
print(70* "-")                                     
print(df.head())
print(70* "-")  
print(df.info())
print(70* "-")  
print(df.isna().sum())
print(70* "-")    
print(df.isna().sum().sum()) 
print(70* "-")                              
print(df["Purchased"].value_counts(normalize=True))  
print(70* "-")  

----------------------------------------------------------------------
(400, 6)
----------------------------------------------------------------------
   UserID  Gender Type  Age  EstimatedSalary  Purchased
0       1    Male    B   24          2800000          0
1       2    Male    A   48          3000000          0
2       3  Female    B   34          7800000          0
3       4  Female    B   36         10700000          0
4       5    Male    D   24         14700000          0
----------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   UserID           400 non-null    int64 
 1   Gender           400 non-null    object
 2   Type             400 non-null    object
 3   Age              400 non-null    int64 
 4   EstimatedSalary  400 non-null    int64 
 5   Purchased        4

In [4]:
#==============================================================================
# Defining features and target
#==============================================================================

X = df.drop(columns=["UserID", "Purchased"])
y = df["Purchased"]

In [ ]:
#==============================================================================
# Train / Test split
#==============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, 
    stratify=y, 
    random_state=42
    )

In [8]:
#==============================================================================
# Preprocessing (definition only, not fitted yet)
#==============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["Age", "EstimatedSalary"]
cat_cols = ["Gender", "Type"]

preprocessor = ColumnTransformer([
    ("numerical", StandardScaler(), num_cols),
    ("categorical", OneHotEncoder(drop="first"), cat_cols),
])

In [ ]:
#==============================================================================
# Building the model
#==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

model = Pipeline([
    ("preprocess", preprocessor),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)),
])

In [11]:
#==============================================================================
# Training
#==============================================================================

model.fit(X_train, y_train)

# How many samples actually shaped the boundary (support vectors per class)
print(model.named_steps["svm"].n_support_)   

[55 54]


In [23]:
#==============================================================================
# Evaluation (baseline model)
#==============================================================================

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print(accuracy_score(y_test, y_pred))     
print(70* "-")     
print(confusion_matrix(y_test, y_pred))  
print(70* "-")     
print(classification_report(y_test, y_pred))

0.9125
----------------------------------------------------------------------
[[46  5]
 [ 2 27]]
----------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.96      0.90      0.93        51
           1       0.84      0.93      0.89        29

    accuracy                           0.91        80
   macro avg       0.90      0.92      0.91        80
weighted avg       0.92      0.91      0.91        80



In [28]:
#==============================================================================
# Hyperparameter tuning
#==============================================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
}

search = GridSearchCV(model, param_grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

print(70* "-")   
print(search.best_params_) 
print(70* "-")    
print(search.best_score_)  
print(70* "-")     

----------------------------------------------------------------------
{'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
----------------------------------------------------------------------
0.8737705627705628
----------------------------------------------------------------------


In [30]:
#==============================================================================
# Final evaluation 
#==============================================================================

best_model = search.best_estimator_
y_pred_final = best_model.predict(X_test)

print(70* "-")   
print(accuracy_score(y_test, y_pred_final))  
print(70* "-")       
print(confusion_matrix(y_test, y_pred_final))  
print(70* "-")     
print(classification_report(y_test, y_pred_final))
print(70* "-")   

----------------------------------------------------------------------
0.925
----------------------------------------------------------------------
[[47  4]
 [ 2 27]]
----------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.96      0.92      0.94        51
           1       0.87      0.93      0.90        29

    accuracy                           0.93        80
   macro avg       0.92      0.93      0.92        80
weighted avg       0.93      0.93      0.93        80

----------------------------------------------------------------------
